In [1]:
# ===========================
# Install dependencies
# ===========================

!pip -q install -U sentence-transformers transformers accelerate

# ===========================
# Imports
# ===========================

import time
import numpy as np
import pandas as pd
import torch

from google.colab import drive
from sentence_transformers import SentenceTransformer

# ===========================
# Mount Google Drive
# ===========================

drive.mount("/content/drive")

# ===========================
# Load dataset
# ===========================

df = pd.read_csv(
    "movies_cleaned.csv"
)

texts = df["embedding_text"].tolist()

print(f"Movies: {len(texts)}")

# ===========================
# GPU check
# ===========================

assert torch.cuda.is_available(), "GPU nije dostupan!"

print(torch.cuda.get_device_name(0))

# ===========================
# Load model
# ===========================

model = SentenceTransformer(
    "Alibaba-NLP/gte-modernbert-base",
    device="cuda",
)

model.max_seq_length = 256

# ===========================
# Warm-up
# ===========================

_ = model.encode(
    texts[:32],
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=False,
)

# ===========================
# Generate embeddings
# ===========================

start = time.perf_counter()

embeddings = model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
)

elapsed = time.perf_counter() - start

print(f"\nEmbedding shape: {embeddings.shape}")
print(f"Time: {elapsed:.2f} s")
print(f"Movies/s: {len(texts) / elapsed:.2f}")

# ===========================
# Save embeddings
# ===========================

output_path = "/content/drive/MyDrive/gte_modernbert_embeddings.npy"

np.save(output_path, embeddings)

print(f"\nSaved to:\n{output_path}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 118.3 MB/s eta 0:00:00
Mounted at /content/drive
Movies: 9967
Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/14.0k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Batches:   0%|          | 0/312 [00:00<?, ?it/s]


Embedding shape: (9967, 768)
Time: 20.00 s
Movies/s: 498.39

Saved to:
/content/drive/MyDrive/gte_modernbert_embeddings.npy
